# Savitsky 1964 — paper calculation audit

This workbook replays the **printed calculations** in Tables 1 and 2, then identifies differences from our engine. It is not a fitted replacement engine. Some printed numbers are internally inconsistent; an exact reproduction of every printed answer is therefore impossible with consistent arithmetic.

Source: [supplied PDF](../../outputs/savitsky/savitsky-1964-reference-2026-10-28/Savitsky-1964.pdf). **Table 1 is PDF page 19 / printed page 89; Table 2 is PDF page 20 / printed page 90.** Read the complete [audit report](../../outputs/savitsky/savitsky-1964-reference-2026-10-28/PAPER-REPLICATION.md).

**Run All** with the same Python environment as `current-hull.ipynb` (`pandas` and IPython). The CLI audit itself needs only the standard library. Edit [paper1964-data.json](paper1964-data.json) to inspect or revise a transcription, then Run All. Every graph reading is explicitly labeled as a reference input; it is not claimed as a computed validation result. The functions below are displayed from the actual imported [Python source](paper1964_audit.py), so there is one calculation implementation.


In [1]:
from pathlib import Path
import importlib
import inspect
import json
import sys
import pandas as pd
from IPython.display import display, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'optimization/savitsky/paper1964-data.json').exists())
sys.path.insert(0, str(ROOT/'optimization/savitsky'))
import paper1964_audit as audit
importlib.reload(audit)
data = audit.load_data()
display(pd.DataFrame([{'input': k, 'value': v, 'note': data['given_notes'].get(k, '')}
                      for k,v in data['given'].items()]))


,input,value,note
0,weight_lbf,60000.00000,
1,beam_ft,14.00000,
2,lcg_ft,29.00000,
3,vcg_ft,2.00000,
4,beta_deg,10.00000,
5,speed_ft_s,67.50000,Paper rounded 40-knot speed retained.
6,gravity_ft_s2,32.20000,
7,density_slug_ft3,1.94000,Inferred from printed lift denominator 0.97 = ...
8,viscosity_ft2_s,0.00001,Approximately inferred from printed Reynolds n...
9,epsilon_deg,4.00000,


## 1. General case: Table 1

The paper tabulates trials at $2^\circ$, $3^\circ$, and $4^\circ$. Graph readings for lift, wetting, friction and pressure center are retained as printed. For every arithmetic row, we evaluate the stated formula using its **printed parent rows**. This exposes local rounding or arithmetic discrepancies instead of allowing one correction to obscure the rest of the table.

The displayed source below contains all operations, including the paper's fixed roughness allowance and friction equation:
$$D_f=\frac{\rho V_m^2\lambda b^2(C_f+0.0004)}{2\cos\beta}.$$
Density is inferred from the example's lift denominator; it is not separately given. The audit reports that uncertainty.


In [2]:
display(Code(inspect.getsource(audit.audit_table1), language='python'))
table1 = pd.DataFrame(audit.audit_table1(data))
display(table1)


def audit_table1(data):
    """Recalculate each non-graph row from its printed upstream operands."""
    g = data['given']
    b, w, beta = g['beam_ft'], g['weight_lbf'], math.radians(g['beta_deg'])
    rho, nu = g['density_slug_ft3'], g['viscosity_ft2_s']
    rows = data['table1_rows']
    output = []
    for j, trim in enumerate(data['table1_trials_deg']):
        r = {item['key']: item['printed'][j] for item in rows}
        t = math.radians(trim)
        calculations = {
            'tau_power': trim**1.1,
            'lift_ratio': r['cl0']/r['tau_power'],
            're': r['vm_ft_s']*r['lambda']*b/nu,
            'cf_total': r['cf']+r['dcf'],
            'df_lbf': rho/2*r['vm_ft_s']**2*r['lambda']*b*b*r['cf_total']/math.cos(beta),
            'tan_tau': math.tan(t), 'sin_tau': math.sin(t), 'cos_tau': math.cos(t),
            'pressure_lbf': w*r['tan_tau'],
            'friction_drag_lbf': r['df_lbf']/r['cos_tau'],
            'drag_lbf': r['pressure_lbf']+r['friction_drag_lbf'],
            'cp_ft': r['cp_ratio']*r['lambda']*b,
            'c_ft': g['lcg_ft']-r['cp_ft'],
            'friction_height_ft': b/4*math.tan(beta),
            'a_ft': g['vcg_ft']-r['friction_height_ft'],
            'sin_sum': math.sin(t+math.radians(g['epsilon_deg'])),
            'vertical_factor': 1-r['sin_tau']*r['sin_sum'],
            'normal_arm_ft': r['vertical_factor']*r['c_ft']/r['cos_tau'],
            'thrust_arm_ft': g['f_ft']*r['sin_tau'],
            'combined_arm_ft': r['normal_arm_ft']-r['thrust_arm_ft'],
            'normal_moment_lbft': w*r['combined_arm_ft'],
            'friction_thrust_arm_ft': r['a_ft']-g['f_ft'],
            'friction_moment_lbft': r['df_lbf']*r['friction_thrust_arm_ft'],
            'moment_lbft': r['normal_moment_lbft']+r['friction_moment_lbft'],
        }
        for item in rows:
            key = item['key']
            note = 'Recomputed from printed parent rows; no hidden rounding.'
            if key not in calculations:
                note = 'Reference input: graph reading or stated friction/roughness choice; not independently verified.'
            if key in {'re', 'df_lbf'}:
                note += ' Uses inferred water properties; see given_notes in JSON.'
            output.append(audit_row(1,item['row'],key,r[key],calculations.get(key),note,trim))
    return output

,table,row,trial_trim_deg,quantity,printed,calculated,difference,note
0,1,1,2,tau_power,2.1400,2.143547,3.546925e-03,Recomputed from printed parent rows; no hidden...
1,1,2,2,cl0,0.0850,NaN,NaN,Reference input: graph reading or stated frict...
2,1,3,2,lift_ratio,0.0397,0.039720,1.962617e-05,Recomputed from printed parent rows; no hidden...
3,1,4,2,lambda,3.8500,NaN,NaN,Reference input: graph reading or stated frict...
4,1,5,2,vm_ft_s,67.0000,NaN,NaN,Reference input: graph reading or stated frict...
...,...,...,...,...,...,...,...,...
85,1,26,4,combined_arm_ft,10.7300,10.665100,-6.490000e-02,Recomputed from printed parent rows; no hidden...
86,1,27,4,normal_moment_lbft,645000.0000,643800.000000,-1.200000e+03,Recomputed from printed parent rows; no hidden...
87,1,28,4,friction_thrust_arm_ft,0.8900,0.890000,-1.110223e-16,Recomputed from printed parent rows; no hidden...
88,1,29,4,friction_moment_lbft,3350.0000,3346.400000,-3.600000e+00,Recomputed from printed parent rows; no hidden...


## 2. General-case interpolation and final outputs

Using the printed moment residuals:
$$q=\frac{149960}{149960+336600},\quad \tau=2^\circ+q(1^\circ).$$
The paper rounds trim to $2.3^\circ$, then uses a fraction of 0.3 to interpolate other values. With the table endpoints,
$$D=9434-(9434-8304)(0.3)=9095\;\mathrm{lbf}.$$
The printed equation starts with **9424**, which instead gives 9085 lbf. For wetting:
$$\lambda=3.85-(3.85-2.60)(0.3)=3.475,$$
while the paper prints **3.29** and uses that value for its following length calculations. This is not explained by ordinary rounding.


In [3]:
display(Code(inspect.getsource(audit.replay_table1_summary), language='python'))
summary = audit.replay_table1_summary(data)
display(pd.DataFrame([{'quantity':k,'recalculated':v} for k,v in summary.items()]))
display(pd.DataFrame([{'quantity':k,'paper_printed':v} for k,v in data['table1_summary'].items()]))


def replay_table1_summary(data):
    """Perform the printed interpolation, keeping the rounded 0.3 fraction explicit."""
    g = data['given']
    r = {x['key']: x['printed'] for x in data['table1_rows']}
    s = data['table1_summary']
    exact_fraction = -r['moment_lbft'][0]/(r['moment_lbft'][1]-r['moment_lbft'][0])
    fraction = 0.3  # Paper rounds trim to 2.3 degrees before subsequent interpolation.
    interp = lambda key: r[key][0]+fraction*(r[key][1]-r[key][0])
    corrected_lambda = interp('lambda')
    half_gap = g['beam_ft']/(2*math.pi)*math.tan(math.radians(g['beta_deg']))/math.tan(math.radians(s['trim_deg']))
    return {
        'unrounded_interpolation_fraction': exact_fraction,
        'unrounded_interpolated_trim_deg': 2+exact_fraction,
        'rounded_trim_deg': round(2+exact_fraction,1),
        'fraction_used_in_printed_followup': fraction,
        'drag_lbf': interp('drag_lbf'),
        'drag_lbf_from_literal_printed_9424': 9424-(9434-8304)*fraction,
        'ehp_from_printed_drag_and_speed': s['drag_lbf']*g['speed_ft_s']/550,
        'lambda_from_printed_interpolation': corrected_lambda,
        'keel_ft_using_printed_lambda': s['lambda']*g['beam_ft']+half_gap,
        'chine_ft_using_printed_lambda': s['lambda']*g['beam_ft']-half_gap,
        'keel_ft_using_recalculated_lambda': corrected_lambda*g['beam_ft']+half_gap,
        'chine_ft_using_recalculated_lambda': corrected_lambda*g['beam_ft']-half_gap,
        'draft_ft_using_printed_keel_and_sine': s['keel_ft']*math.sin(math.radians(s['trim_deg'])),
    }

,quantity,recalculated
0,unrounded_interpolation_fraction,0.308205
1,unrounded_interpolated_trim_deg,2.308205
2,rounded_trim_deg,2.300000
3,fraction_used_in_printed_followup,0.300000
4,drag_lbf,9095.000000
5,drag_lbf_from_literal_printed_9424,9085.000000
6,ehp_from_printed_drag_and_speed,1116.204545
7,lambda_from_printed_interpolation,3.475000
8,keel_ft_using_printed_lambda,55.842016
9,chine_ft_using_printed_lambda,36.277984


,quantity,paper_printed
0,trim_deg,2.30
1,drag_lbf,9095.00
2,ehp,1115.00
3,lambda,3.29
4,keel_ft,55.90
5,chine_ft,36.10
6,draft_ft,2.24


## 3. Simplified case: Table 2 and the 2.23° trim

Table 2 assumes $a=f=c=\epsilon=0$: all force lines pass through the CG. It is a distinct case, not the ordinary engine with only its thrust angle changed.

The table first evaluates $\tau^{1.1}=0.085/0.035$ and prints **2.42**. Using that rounded intermediate:
$$\tau=2.42^{1/1.1}=2.23317656^\circ\longrightarrow2.23^\circ.$$
Keeping the quotient at full precision gives a different rounded answer. This is an identifiable rounding effect. The 0.085 and 0.035 coefficients are graph readings, not independent equation-solver outputs.


In [4]:
display(Code(inspect.getsource(audit.audit_table2), language='python'))
table2_rows, table2_extra = audit.audit_table2(data)
display(pd.DataFrame(table2_rows))
display(pd.DataFrame([{'quantity':k,'value':v} for k,v in table2_extra.items()]))


def audit_table2(data):
    """Audit simplified-case arithmetic, retaining its graph inputs and rounded rows."""
    g = data['given']
    r = {item['key']: item['printed'] for item in data['table2_rows']}
    b, w = g['beam_ft'], g['weight_lbf']
    rho, nu = g['density_slug_ft3'], g['viscosity_ft2_s']
    cos_beta = math.cos(math.radians(g['beta_deg']))
    calculations = {
        'cp_over_beam': g['lcg_ft']/b,
        'tau_power': r['cl0']/r['lift_ratio'],
        'trim_deg': r['tau_power']**(1/1.1),
        'tan_tau': math.tan(math.radians(r['trim_deg'])),
        'pressure_lbf': w*r['tan_tau'],
        'lambda_b2_ft2': r['lambda']*b*b,
        're': r['vm_ft_s']*r['lambda']*b/nu,
        'cf_total': r['cf']+r['dcf'],
        'df_lbf': rho/2*r['vm_ft_s']**2*r['lambda_b2_ft2']*r['cf_total']/cos_beta,
        'friction_drag_lbf': r['df_lbf']/math.cos(math.radians(r['trim_deg'])),
        'drag_lbf': r['pressure_lbf']+r['friction_drag_lbf'],
        'ehp': r['drag_lbf']*g['speed_ft_s']/550,
        'porpoising_axis': math.sqrt(.069/2),
    }
    output = []
    for item in data['table2_rows']:
        key = item['key']
        note = 'Recomputed from printed parent rows; no hidden rounding.'
        if key not in calculations:
            note = 'Reference input from graph or stated coefficient; not independently verified.'
        if key in {'re','df_lbf'}:
            note += ' Uses inferred water properties.'
        output.append(audit_row(2,item['row'],key,r[key],calculations.get(key),note))
    # An exact carry-through differs from the paper's intermediate rounding.
    precise_trim = (r['cl0']/r['lift_ratio'])**(1/1.1)
    implied_density = 2*r['df_lbf']*cos_beta/(r['vm_ft_s']**2*r['lambda_b2_ft2']*r['cf_total'])
    return output, {'trim_without_intermediate_rounding_deg': precise_trim,
                    'trim_from_printed_2_42_deg': calculations['trim_deg'],
                    'density_implied_by_printed_friction_slug_ft3': implied_density}

,table,row,trial_trim_deg,quantity,printed,calculated,difference,note
0,2,1,None,cl0,8.500000e-02,NaN,NaN,Reference input from graph or stated coefficie...
1,2,2,None,cp_over_beam,2.070000e+00,2.071429e+00,1.428571e-03,Recomputed from printed parent rows; no hidden...
2,2,3,None,lambda,3.450000e+00,NaN,NaN,Reference input from graph or stated coefficie...
3,2,4,None,lift_ratio,3.500000e-02,NaN,NaN,Reference input from graph or stated coefficie...
4,2,5,None,tau_power,2.420000e+00,2.428571e+00,8.571429e-03,Recomputed from printed parent rows; no hidden...
5,2,6,None,trim_deg,2.230000e+00,2.233177e+00,3.176556e-03,Recomputed from printed parent rows; no hidden...
6,2,7,None,tan_tau,3.900000e-02,3.894051e-02,-5.949292e-05,Recomputed from printed parent rows; no hidden...
7,2,8,None,pressure_lbf,2.340000e+03,2.340000e+03,0.000000e+00,Recomputed from printed parent rows; no hidden...
8,2,9,None,lambda_b2_ft2,6.750000e+02,6.762000e+02,1.200000e+00,Recomputed from printed parent rows; no hidden...
9,2,10,None,vm_ft_s,6.690000e+01,NaN,NaN,Reference input from graph or stated coefficie...


,quantity,value
0,trim_without_intermediate_rounding_deg,2.240366
1,trim_from_printed_2_42_deg,2.233177
2,density_implied_by_printed_friction_slug_ft3,2.003972


## 4. Graph readings versus explicit equations

This checks the printed graph readings against the corresponding analytic lift and pressure-center relations, without pretending that graph readings carry full numerical precision. It does not retune any constants or solve a new equilibrium. Reynolds number and density reconstruction remain visible above.


In [5]:
display(Code(inspect.getsource(audit.equation_spot_checks), language='python'))
display(pd.DataFrame(audit.equation_spot_checks(data)))


def equation_spot_checks(data):
    """Compare graph-derived lift/pressure readings with explicit printed equations.

Equations 15/16 use the load coefficient weight/(rho/2 V² b²), as the
worked tables do, rather than replacing that coefficient with our engine's
coupled vertical-force solution. This distinction is intentional.
"""
    g = data['given']
    cv = g['speed_ft_s']/math.sqrt(g['gravity_ft_s2']*g['beam_ft'])
    cl_beta_load = g['weight_lbf']/(g['density_slug_ft3']/2*g['speed_ft_s']**2*g['beam_ft']**2)
    output = []
    for j, trim in enumerate(data['table1_trials_deg']):
        lam = next(x for x in data['table1_rows'] if x['key']=='lambda')['printed'][j]
        cl0 = trim**1.1*(.012*math.sqrt(lam)+.0055*lam**2.5/cv**2)
        cb = cl0-.0065*g['beta_deg']*cl0**.6
        cp = .75-1/(5.21*(cv/lam)**2+2.39)
        output.append(dict(trial_trim_deg=trim,lambda_from_graph=lam,cv=cv,
                           clbeta_from_loading=cl_beta_load,cl0_from_eq15=cl0,
                           clbeta_from_eq16=cb,cp_ratio_from_eq28=cp))
    return output

,trial_trim_deg,lambda_from_graph,cv,clbeta_from_loading,cl0_from_eq15,clbeta_from_eq16,cp_ratio_from_eq28
0,2,3.85,3.179156,0.069265,0.084397,0.069649,0.581722
1,3,2.60,3.179156,0.069265,0.084650,0.069876,0.651764
2,4,1.86,3.179156,0.069265,0.086995,0.071977,0.693216


## 5. Differences from our current engine

| Item | Paper calculation | Current engine |
|---|---|---|
| Lift and wetting | Weight-based load coefficient and graph readings | Coupled force/moment solution |
| Friction coefficient | Schoenherr | ITTC 1957 |
| Roughness increment | Fixed 0.0004 | Zero or Townsin from dimensional roughness |
| Speed squared in friction force | Bottom-flow speed | Forward speed; bottom-flow speed only in Reynolds number |
| Friction force height | b tan(beta) / 4 in Table 1 | Area-weighted height |
| Equilibrium | Approximate Eq.35, trial table, linear interpolation, intermediate rounding | Continuous residual bisection |
| Simplified Table 2 | Friction and thrust lines pass through CG | Requires a distinct force arrangement |

The paper's weight*tan(trim) and Df/cos(trim) resistance decomposition also has a different meaning from the engine's separately reported horizontal force components in the general thrust case.

The earlier modern-roughness result of 8997 lbf is close to 9095 lbf, but it does not constitute an exact reproduction. The corrected paper interpolation gives wetting much closer to the engine than the paper's printed 3.29 suggests. The physical accuracy of the method for our non-prismatic hull remains a separate question.


In [6]:
# Source-based arithmetic checks, not a claim that every paper value matches.
assert summary['drag_lbf'] == 9095
assert summary['drag_lbf_from_literal_printed_9424'] == 9085
assert summary['lambda_from_printed_interpolation'] == 3.475
assert round(summary['rounded_trim_deg'], 1) == data['table1_summary']['trim_deg']
assert round(table2_extra['trim_from_printed_2_42_deg'], 2) == 2.23
assert len(table1) == 90 and len(table2_rows) == 19
print('PASS: complete numeric row audit; general-case drag interpolation; simplified-case rounded trim.')
print('Confirmed discrepancies are preserved in the tables, not suppressed by tolerances.')


PASS: complete numeric row audit; general-case drag interpolation; simplified-case rounded trim.
Confirmed discrepancies are preserved in the tables, not suppressed by tolerances.


## 6. Optional refresh of the audit report

The following disabled cell can regenerate the report, JSON and CSV in the existing reference output folder. The original current-hull runs and production model are unchanged.


In [7]:
WRITE_REPORT = False
if WRITE_REPORT:
    audit.write_audit()
    print('Updated paper audit report, JSON and CSV.')
